In [ ]:
from pyspark.sql.functions import current_timestamp, col
ambiente = 'dev'

In [ ]:
file = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("cloudFiles.useIncrementalListing", "true")
        .load(f"/Volumes/cor_{ambiente}/bronze/data/landing/")
)

In [ ]:
df_bronze_raw_swell_metrics = file.select(
    col("value").alias("data"),
    col("_metadata.file_path").alias("source_file"),
    current_timestamp().alias("ingestion_timestamp")
)

In [ ]:
(
    df_bronze_raw_swell_metrics.writeStream
        .format("delta")
        .trigger(availableNow=True)
        .option("checkpointLocation", f"/Volumes/cor_{ambiente}/bronze/data/checkpoints/raw_swell_metrics")
        .toTable(f"cor_{ambiente}.bronze.raw_swell_metrics")
)